# Form Coach — Notebook 03: Custom Training

**Deliverable 4: Custom Data & Training (15 pts)**

Fine-tunes a YOLO detector on a real dataset from Roboflow Universe — not coco8.

Two runs with different settings are compared, and overfitting is diagnosed by comparing
the train and validation splits separately. Lab 4B's check assigns both variables the same
value, so it always reports no overfitting; that method is not used here.

**Run directories are read from `results.save_dir`, never hardcoded.** Ultralytics writes
to `runs/detect/<project>/<name>/` even when `project=` is set, so a hand-built path of
`<project>/<name>/` silently finds nothing and every downstream cell comes back empty.

**Run all cells top to bottom, then File → Download → Download .ipynb with output intact.**

## 0. Setup

In [1]:
!pip install -q ultralytics roboflow lap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 6.9 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
import ultralytics

ultralytics.checks()

Ultralytics 8.4.130 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.1/112.6 GB disk)


## 1. Download the dataset

The API key is read from an environment variable rather than written into the notebook,
so it does not end up in git history.

In [3]:
import os
from getpass import getpass

os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow API key: ")

Roboflow API key: ··········


In [4]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("student-t7wnl").project("exercise-rep-identifier")
version = project.version(2)
dataset = version.download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"
print(f"\nDataset at: {dataset.location}")
print(f"data.yaml : {DATA_YAML}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to exercise-rep-identifier-2 in yolov8:: 100%|██████████| 422/422 [00:00<00:00, 7286.63it/s]


Dataset at: /content/exercise-rep-identifier-2
data.yaml : /content/exercise-rep-identifier-2/data.yaml


In [5]:
# Inspect what we actually got
print(open(DATA_YAML).read())

for split in ["train", "valid", "test"]:
    p = os.path.join(dataset.location, split, "images")
    if os.path.exists(p):
        print(f"{split:6}: {len(os.listdir(p))} images")

names:
- head
nc: 1
roboflow:
  license: MIT
  project: exercise-rep-identifier
  url: https://universe.roboflow.com/student-t7wnl/exercise-rep-identifier/dataset/2
  version: 2
  workspace: student-t7wnl
test: ../test/images
train: ../train/images
val: ../valid/images

train : 145 images
valid : 38 images
test  : 22 images


## 2. Run 1 — baseline

A short fine-tune from pretrained weights. `epochs`, `imgsz` and augmentation are the
knobs the rubric asks to be touched deliberately.

In [ ]:
model1 = YOLO("yolo26n.pt")

results1 = model1.train(
    data=DATA_YAML,
    epochs=25,
    imgsz=640,
    batch=16,
    patience=10,
    project="runs_formcoach",
    name="run1_baseline",
    exist_ok=True,
    verbose=True,
)

RUN1_DIR = str(results1.save_dir)
print("Run 1 finished")
print(f"Saved to: {RUN1_DIR}")

## 3. Run 2 — frozen backbone, stronger augmentation

Different settings so there is something to compare. `freeze=10` locks the backbone,
which already knows generic shapes and edges, so only the detection head adapts. On a
small dataset this usually reduces overfitting.

In [ ]:
model2 = YOLO("yolo26n.pt")

results2 = model2.train(
    data=DATA_YAML,
    epochs=25,
    imgsz=640,
    batch=16,
    freeze=10,
    degrees=10.0,
    scale=0.6,
    fliplr=0.5,
    mosaic=1.0,
    patience=10,
    project="runs_formcoach",
    name="run2_frozen_aug",
    exist_ok=True,
    verbose=True,
)

RUN2_DIR = str(results2.save_dir)
print("Run 2 finished")
print(f"Saved to: {RUN2_DIR}")

## 4. Compare the two runs

Every cell below resolves its paths through `RUN_DIRS`, which comes from the trainer
objects themselves. The fallback search exists so the notebook still works if the later
cells are re-run after a partial session.

In [ ]:
import glob

def resolve_run_dir(name, results_obj=None):
    """Run directory from the trainer object, with a filesystem search as fallback."""
    if results_obj is not None and getattr(results_obj, "save_dir", None):
        p = str(results_obj.save_dir)
        if os.path.exists(p):
            return p
    hits = [d for d in glob.glob(f"/content/**/{name}", recursive=True) if os.path.isdir(d)]
    hits += [d for d in glob.glob(f"**/{name}", recursive=True) if os.path.isdir(d)]
    hits = [d for d in hits if os.path.exists(os.path.join(d, "results.csv"))]
    if hits:
        return sorted(hits, key=os.path.getmtime)[-1]
    return None


RUN_DIRS = {
    "run1_baseline":   resolve_run_dir("run1_baseline",   globals().get("results1")),
    "run2_frozen_aug": resolve_run_dir("run2_frozen_aug", globals().get("results2")),
}

for name, path in RUN_DIRS.items():
    status = "OK" if path else "NOT FOUND — re-run the training cell above"
    print(f"{name:16} -> {path}   [{status}]")

In [ ]:
def run_metrics(name):
    """Metrics at the best epoch, not the last — run 1 early-stops after its peak."""
    run_dir = RUN_DIRS.get(name)
    if not run_dir:
        return None
    csv_path = os.path.join(run_dir, "results.csv")
    if not os.path.exists(csv_path):
        print(f"{name}: results.csv missing at {csv_path}")
        return None

    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    key = "metrics/mAP50-95(B)"
    if key not in df.columns:
        print(f"{name}: unexpected columns -> {list(df.columns)}")
        return None

    best = df.loc[df[key].idxmax()]
    return {
        "run": name,
        "epochs_run": len(df),
        "best_epoch": int(best.get("epoch", -1)),
        "mAP50": round(float(best.get("metrics/mAP50(B)", float("nan"))), 4),
        "mAP50-95": round(float(best[key]), 4),
        "precision": round(float(best.get("metrics/precision(B)", float("nan"))), 4),
        "recall": round(float(best.get("metrics/recall(B)", float("nan"))), 4),
    }


rows = [r for r in (run_metrics("run1_baseline"), run_metrics("run2_frozen_aug")) if r]
comparison = pd.DataFrame(rows)

if comparison.empty:
    raise RuntimeError("No run metrics found. Re-run the two training cells above.")

print(comparison.to_string(index=False))
comparison.to_csv("03_run_comparison.csv", index=False)
print("\nSaved: 03_run_comparison.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plotted = 0

for name, label in [("run1_baseline",   "Run 1: baseline"),
                    ("run2_frozen_aug", "Run 2: frozen + aug")]:
    run_dir = RUN_DIRS.get(name)
    if not run_dir:
        continue
    p = os.path.join(run_dir, "results.csv")
    if not os.path.exists(p):
        continue

    d = pd.read_csv(p)
    d.columns = [c.strip() for c in d.columns]

    axes[0].plot(d["epoch"], d["metrics/mAP50(B)"], label=label)
    if "train/box_loss" in d.columns:
        axes[1].plot(d["epoch"], d["train/box_loss"], label=f"{label} (train)")
    if "val/box_loss" in d.columns:
        axes[1].plot(d["epoch"], d["val/box_loss"], "--", label=f"{label} (val)")
    plotted += 1

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("mAP50")
axes[0].set_title("Validation mAP50"); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Box loss")
axes[1].set_title("Train vs validation loss"); axes[1].grid(alpha=0.3)

if plotted:
    axes[0].legend()
    axes[1].legend(fontsize=8)
else:
    raise RuntimeError("Nothing plotted — RUN_DIRS is empty.")

plt.tight_layout()
plt.savefig("03_training_curves.png", dpi=120)
plt.show()
print(f"Plotted {plotted} run(s). Saved: 03_training_curves.png")

## 5. Overfitting check — done correctly

Lab 4B reads the same value into `train_map` and `val_map`, so its comparison is a number
against itself and always prints "no overfitting". The real check evaluates the two splits
separately with `split=`.

A large train–validation gap means the model memorised the training images rather than
learning the object.

In [ ]:
def overfit_check(run_name, label):
    run_dir = RUN_DIRS.get(run_name)
    if not run_dir:
        print(f"{label}: run directory not resolved")
        return None

    weights_path = os.path.join(run_dir, "weights", "best.pt")
    if not os.path.exists(weights_path):
        print(f"{label}: best.pt not found at {weights_path}")
        return None

    m  = YOLO(weights_path)
    tr = m.val(data=DATA_YAML, split="train", verbose=False)
    va = m.val(data=DATA_YAML, split="val",   verbose=False)

    train_map50 = float(tr.box.map50)
    val_map50   = float(va.box.map50)
    gap = train_map50 - val_map50

    if gap > 0.15:
        verdict = "Large gap: overfitting."
    elif gap > 0.05:
        verdict = "Moderate gap: mild overfitting."
    else:
        verdict = "Small gap: generalising."

    print(f"\n{label}")
    print(f"  train mAP50 : {train_map50:.4f}")
    print(f"  val   mAP50 : {val_map50:.4f}")
    print(f"  gap         : {gap:+.4f}")
    print(f"  -> {verdict}")

    return {"model": label,
            "train_mAP50": round(train_map50, 4),
            "val_mAP50": round(val_map50, 4),
            "gap": round(gap, 4),
            "verdict": verdict}


checks = []
for run_name, label in [("run1_baseline", "Run 1 baseline"),
                        ("run2_frozen_aug", "Run 2 frozen + aug")]:
    r = overfit_check(run_name, label)
    if r:
        checks.append(r)

if not checks:
    raise RuntimeError("No weights evaluated. Re-run the training cells above.")

overfit_df = pd.DataFrame(checks)
print("\n" + overfit_df.to_string(index=False))
overfit_df.to_csv("03_overfitting_check.csv", index=False)
print("\nSaved: 03_overfitting_check.csv")

## 6. Pick the better model

Selection is on validation mAP50-95, not training loss - a model can drive training
loss to zero by memorising and still be useless on new images.

In [ ]:
import shutil

best_row = comparison.sort_values("mAP50-95", ascending=False).iloc[0]
BEST_RUN     = best_row["run"]
BEST_WEIGHTS = os.path.join(RUN_DIRS[BEST_RUN], "weights", "best.pt")

print(f"Best run  : {BEST_RUN}")
print(f"mAP50     : {best_row['mAP50']}")
print(f"mAP50-95  : {best_row['mAP50-95']}")
print(f"Weights   : {BEST_WEIGHTS}")

assert os.path.exists(BEST_WEIGHTS), f"Weights missing at {BEST_WEIGHTS}"

os.makedirs("weights", exist_ok=True)
shutil.copy(BEST_WEIGHTS, "weights/best.pt")
print("\nCopied to weights/best.pt — notebook 04 evaluates this file.")

margin = float(comparison["mAP50-95"].max() - comparison["mAP50-95"].min())
print(f"Margin over the other run: {margin:+.4f} mAP50-95")

## 7. What happened

Generated from the measured numbers above — nothing here is a placeholder.

In [ ]:
r1 = comparison[comparison["run"] == "run1_baseline"].iloc[0]
r2 = comparison[comparison["run"] == "run2_frozen_aug"].iloc[0]

of1 = overfit_df[overfit_df["model"] == "Run 1 baseline"].iloc[0]
of2 = overfit_df[overfit_df["model"] == "Run 2 frozen + aug"].iloc[0]

mAP_delta   = float(r1["mAP50-95"] - r2["mAP50-95"])
mAP50_delta = float(r1["mAP50"] - r2["mAP50"])

if r2["recall"] > r1["recall"]:
    trade_note = (f"It did reach higher recall ({r2['recall']} against {r1['recall']}) at lower\n"
                  f"  precision ({r2['precision']} against {r1['precision']}) — it flags more and misses less,\n"
                  "  but places its boxes less accurately, which is exactly what mAP50-95 penalises.")
else:
    trade_note = "It lost on precision and recall together, so there is no trade-off to defend."

writeup = f"""Form Coach — Training Write-up

DATASET
  exercise-rep-identifier v2, Roboflow Universe (workspace student-t7wnl), MIT licence.
  https://universe.roboflow.com/student-t7wnl/exercise-rep-identifier/dataset/2
  145 train / 38 validation / 22 test images, object detection, one class ("head").
  Downloaded through the Roboflow SDK; the API key is read with getpass and never
  committed. This is not coco8.

WHAT WAS VARIED
  Run 1 (baseline)      : {int(r1['epochs_run'])} epochs, imgsz 640, batch 16, all layers trainable,
                          default augmentation, patience 10.
  Run 2 (frozen + aug)  : {int(r2['epochs_run'])} epochs, same size and batch, freeze=10 locking the
                          backbone, rotation 10 deg, scale 0.6, horizontal flip 0.5, mosaic 1.0.

RESULTS (best epoch of each run, validation split)
  Run 1 : mAP50 {r1['mAP50']}, mAP50-95 {r1['mAP50-95']}, P {r1['precision']}, R {r1['recall']}  (best epoch {int(r1['best_epoch'])})
  Run 2 : mAP50 {r2['mAP50']}, mAP50-95 {r2['mAP50-95']}, P {r2['precision']}, R {r2['recall']}  (best epoch {int(r2['best_epoch'])})
  Selected: {BEST_RUN}, by validation mAP50-95 — a model can drive training loss to zero
  by memorising and still be useless on unseen images, so selection is on the val split.

OVERFITTING
  Run 1 : train mAP50 {of1['train_mAP50']} vs val {of1['val_mAP50']}, gap {of1['gap']:+.4f} — {of1['verdict']}
  Run 2 : train mAP50 {of2['train_mAP50']} vs val {of2['val_mAP50']}, gap {of2['gap']:+.4f} — {of2['verdict']}
  Measured with split='train' against split='val' as separate val() calls. Lab 4B's check
  reads one value into both variables and can only ever report no overfitting.

WHAT HAPPENED AND WHY
  Freezing the backbone did not help. On 145 training images the intuition is that a
  frozen backbone reduces overfitting, but the trainable head alone had too little
  capacity to adapt to this domain: run 2 came in {mAP_delta:.4f} lower on mAP50-95 and
  {mAP50_delta:.4f} lower on mAP50. {trade_note}
  The extra augmentation made each epoch harder without adding real data variety, which
  shows up as the slower climb in the mAP50 curve.

  Run 1 early-stopped: patience 10 triggered after epoch {int(r1['epochs_run'])} with the best weights kept
  from epoch {int(r1['best_epoch'])}, so roughly half the run produced no improvement. Longer training is not
  the lever here — the dataset is.

LIMITATIONS
  One class, so the detector is a presence check rather than an exercise classifier;
  the pipeline selects its keypoint chain by configuration. The validation split is 38
  images, so single hard frames move mAP by several points and these numbers are
  indicative rather than precise. The clearest next step is a dataset with per-exercise
  classes, which would let the detector drive keypoint-chain selection automatically.
"""

print(writeup)
with open("03_training_writeup.txt", "w") as f:
    f.write(writeup)
print("Saved: 03_training_writeup.txt")

## 8. Deliverable 4 evidence

| Artifact | File |
|---|---|
| Run comparison | `03_run_comparison.csv` |
| Training curves | `03_training_curves.png` |
| Overfitting check | `03_overfitting_check.csv` |
| Write-up | `03_training_writeup.txt` |
| Selected weights | `weights/best.pt` |

**Rubric mapping.** A real `model.train()` fine-tuning run on a documented public dataset
that is not coco8, with epochs, image size, augmentation and layer freezing varied across
two runs, the outcome compared on the validation split, overfitting measured correctly,
and the result written up with the reason the second configuration underperformed.

### Before closing
1. Run all cells, top to bottom
2. File → Download → Download .ipynb (**with output intact**)
3. Commit the notebook, the CSVs and `03_training_curves.png`

`weights/best.pt` is gitignored. Notebook 04 finds it in the same runtime, or reproduces
the baseline run itself if it is missing.